### Section B (TOTAL: 70 marks)

In this part of the coursework, you will create a set of Python classes to represent commuters travelling on a metro system such as the London Underground.

Parts 1–3 should be implemented in a module named tube_utils.py, which extends the classes provided in network_utils.py. Parts 4–7 should be completed in this Jupyter notebook.

Make sure that all relevant files are saved in the same directory as this notebook. You should not use absolute file paths at any stage.



### Part 1 
[6 marks]

The network_utils.py module contains several class definitions from Lab 10.

Your task is to create a new class, Station, which inherits from the Vertex class. The Station class should store information about each station, including its ID, name, and line.

Note that the same physical station appearing on different lines should be represented as distinct vertices. For example, Oxford Circus on the Bakerloo line and Oxford Circus on the Central line should be treated as separate Station objects.

### Part 2 
[12 marks]
 
Create a class called `TubeNetwork`, which inherits from the `SymmetricNetwork` class.

The `TubeNetwork` class should include:
- method that takes two Station objects and a connection time as input, and adds an appropriate edge between them
- a method that takes two Station objects as input and returns the distance of the shortest path between them. You may assume that all Stations are connected.

### Part 3 
[12 marks]

Create a class called `Commuter`.

The class should be initialised with the ID of a starting station, the ID of an end station, and a `TubeNetwork` object, representing the transportation network.  When a Commuter object is created, it should automatically calculate and store the time taken for the shortest route between the two stations. This should handle the case where the starting or end stations are on more than one line.

The class should include:
- a method that takes as input a `TubeNetwork` object and updates the stored travel time; and
- a method that returns the time taken for the shortest route between the start and end stations.

If either the start or end station is not present in the network, the method should raise a custom error, `StationNotFoundError`, which you should also define. The error message should clearly indicate which station is missing from the network.

### Part 4 
[12 marks]

The folder 'underground' contains data describing stations and connections on the London Underground.

Write code to initialise a `TubeNetwork` object using this information. Your code should:
- read data from the CSV files provided in the folder;
- create Station objects for each distinct vertex in the network; and
- add edges to the network representing travel times between adjacent stations and transfers between lines at the same station.

The resulting `TubeNetwork` object should be ready to use for calculations in later parts of this section.

In [4]:
# YOUR CODE HERE

from network_utils import Edge
from network_utils import Vertex
from network_utils import Network
from network_utils import SymmetricNetwork
from tube_utils import TubeNetwork
from tube_utils import Station
from tube_utils import Commuter
from tube_utils import StationNotFoundError

tubenetwork = TubeNetwork()
stations = {} #key: (station_id, line)



def get_station(station_id, line):
    """
    Retrieves a Station object for a given ID and line. 
    If it doesn't exist, creates it and adds it to the network.
    """
    
    key = (station_id, line)
 
    if key not in stations:
        new_station = Station(vertex_id=key, name=station_id, line=line)
        stations[key] = new_station
        tubenetwork.add_vertex(new_station)


    return stations[key]

#1. Build the connection between different lines
lines = [
    "victoria", "piccadilly", "northern", 
    "metropolitan", "jubilee", "hammersmith_city",
    "district", "central", "bakerloo"
]

for line_name in lines:
    with open(f"UG_{line_name}.csv", 'r') as f:
    
        heading = next(f) # Skips the heading
        
        for row in f: #Iterate over each row in the csv

            # devided by "," and read by each lines
            parts = row.strip().split(',')
            # if there is not enough information on this line, skip
            if len(parts) != 3: 
                continue
            start_id, end_id, time_between = parts
            
            start_station = get_station(start_id, line_name)
            end_station = get_station(end_id, line_name)
            time = float(time_between)

            #add_connection includes add_edge and Edge()
            tubenetwork.add_connection(start_station, end_station, time) 
            
       
        
# 2. Build transfer between same station but different lines
# Map same station with different lines together
physical_stations_map = {}

# Run through all the stations we just built above
for (station_id, line), station_obj in stations.items():
    if station_id not in physical_stations_map:
        physical_stations_map[station_id] = []
    physical_stations_map[station_id].append(station_obj)


# Get transfer time between same station but different lines
with open("stations.csv", 'r') as r:
    # Skip the heading
    next(r)
    
    for row in r:
        parts = row.strip().split(',')
        if len(parts) != 3:
            continue
        
        stat_id, stat_name, trsfr_min_str = parts
        transfer_time = float(trsfr_min_str)
        
        # Check if there are multiple lines in this station
        if stat_id in physical_stations_map:
            station_nodes = physical_stations_map[stat_id]
            
            # If there is just one line in this station, no need for transfer
            if len(station_nodes) > 1:
                # Go through all the lines on this same station, connecting them together
                # For example if it is Line A, Line B and Line C, we need to connect
                # Line A-B, Line A-C, LineB-C
                for i in range(len(station_nodes)):
                    for j in range(i + 1, len(station_nodes)):
                        node_1 = station_nodes[i]
                        node_2 = station_nodes[j]
                        
                        tubenetwork.add_connection(node_1, node_2, transfer_time)


        


### Part 5 
[6 marks]

The file `commuters.csv` contains data on a set of commuters in London.

Write code to read this file and create a collection of Commuter objects.

Some commuters in the file will have invalid start or end stations that are not present in the TubeNetwork. Use a try-except block to handle these cases appropriately.


In [5]:
# YOUR CODE HERE

# ==========================================
# Part 5: Reading Commuters Data
# ==========================================

commuters = []

with open("commuters.csv", "r") as g:
    next(g) # Skip the headings
    
    for row in g:
        parts = row.strip().split(',')
        if len(parts) != 5:
            continue
            
        com_id, sta_id, sta_name, en_id, en_name = parts

        try:
            # Initialise commuter，if it is invalid，remind with error
            c = Commuter(start_station_ID=sta_id, end_station_ID=en_id, tube_network=tubenetwork)
            
            # if only the commuter information is valid
            commuters.append(c)
            # Print the result for clarification 
            print(f"Commuter {com_id}: {sta_name} -> {en_name} takes {c.get_travel_time()} mins.")
            
        except StationNotFoundError:
            # pass if error
            pass

print(f"\n Successfully loaded {len(commuters)} valid commuters.")


Commuter 1: Caledonian Road -> Snaresbrook takes 29.0 mins.
Commuter 2: Highgate -> Chiswick Park takes 48.0 mins.
Commuter 3: Latimer Road -> Stratford takes 47.0 mins.
Commuter 4: Upton Park -> Bayswater takes 42.0 mins.
Commuter 5: Loughton -> Hounslow West takes 84.0 mins.
Commuter 6: Rickmansworth -> Edgware takes 60.0 mins.
Commuter 7: South Woodford -> South Kenton takes 71.0 mins.
Commuter 8: Piccadilly Circus -> Euston takes 11.0 mins.
Commuter 9: East Acton -> Highgate takes 44.0 mins.
Commuter 10: Bond Street -> Finsbury Park takes 21.0 mins.
Commuter 11: Northfields -> Royal Oak takes 30.0 mins.
Commuter 12: Hendon Central -> Tufnell Park takes 13.0 mins.
Commuter 13: Eastcote -> Ickenham takes 6.0 mins.
Commuter 14: Tooting Bec -> Hounslow Central takes 62.0 mins.
Commuter 16: Harrow-On-The-Hill -> Stanmore takes 16.0 mins.
Commuter 17: Elm Park -> Oval takes 46.0 mins.
Commuter 18: Bow Road -> Hammersmith (D&P) takes 44.0 mins.
Commuter 19: Ruislip Manor -> Chalfont & Lat

### Part 6 
[8 marks]

Your personal file `station_closed.txt` contains the station ID of a station closed for engineering works. During the closure, line changes at this station are not permitted (travelling through the station on the same line remains allowed).

Write code to:  
1.  Read in the station ID from the file.
2.	Compute the average journey time across all commuters under normal conditions.  
3.	Recompute the average journey time when transfers at station_closed are disallowed.  
4.	Report the difference between these two averages.

You may assume that all journeys remain reachable, i.e. no commuter journeys start or end at the closed station.

In [6]:
# YOUR CODE HERE
# ==========================================
# Part 6: Station Closure Impact Analysis
# ==========================================

# 1. read the closed station from the file
with open("station_closed.txt", "r") as h:
    closed_station = h.read().strip()

# 2. Calculate the average time under normal situation
total_time_normal = 0
count = 0

for c in commuters:
    # already get travel time in Part 5, using it here
    total_time_normal += c.get_travel_time()
    count += 1

avg_normal = total_time_normal / count
print(f"The average normal travel time is {avg_normal:.2f} mins.")

# 3. Stop all connections to this closed station

closed_nodes = [v for v in tubenetwork._vertices if v.name == closed_station]

# Just like Part 4 when we built connection,
# Use for loops to stop the connections to this closed station
if len(closed_nodes) > 1:
    for i in range(len(closed_nodes)):
        for j in range(i + 1, len(closed_nodes)):
            node_1 = closed_nodes[i]
            node_2 = closed_nodes[j]
            # Remove the edge
            tubenetwork.remove_edge(node_1, node_2)

print(f"\n[System] Transfers at station '{closed_station}' have been disabled.")

# 4. Calculate the average time after a station closed
total_time_closed = 0

for c in commuters:
    # Dijkstra used again for the situaion if a station is closed
    c.update_travel_time(tubenetwork) 
    total_time_closed += c.get_travel_time()

avg_closed = total_time_closed / count
print(f"The average travel time with station {closed_station} closed is {avg_closed:} mins.")

# 5. Average time difference
difference = avg_closed - avg_normal
print(f"The difference between average journey times is {difference:.2f} mins.")

The average normal travel time is 40.03 mins.

[System] Transfers at station '237' have been disabled.
The average travel time with station 237 closed is 40.494791666666664 mins.
The difference between average journey times is 0.46 mins.


### Part 7 
[14 marks]

Write a short report (up to 300 words) discussing your solutions to Parts 1–6.

Your report should:
- justify your choice of data structures; and
- explain how you have applied the principles of object-oriented programming in designing your classes;

Finally, suppose you wanted to make the model more realistic by allowing different commuters to have different walking speeds, so that the time taken to change lines or stations varies by commuter. Describe how you could adapt your implementation to reflect this, outlining any changes you would make to your class design or method definitions. Reflect on how your current implementation would facilitate this extension.

### London Underground Network Model: Design Report

1. Data Structures
I modeled the network as a graph to prioritize routing efficiency. Sets are used to guarantee vertex uniqueness, while dictionaries (_adjacencies) provide $O(1)$ lookups crucial for Dijkstra’s algorithm. To handle multi-line stations, I designed a dictionary registry using (station_id, line) tuples as keys. This ensures that the same physical station on different lines is instantiated as distinct vertices, seamlessly enabling realistic transfer connections.

2. Object-Oriented Programming Principles
OOP principles strictly guided the architecture. Inheritance allows Station and TubeNetwork to extend base graph classes, minimizing code duplication. To protect data integrity, I applied Encapsulation: internal structures like _vertices are kept as private/protected attributes, safely modifiable only through public methods like add_connection(). Crucially, I utilized Polymorphism by overriding the __lt__ magic method within the Station class. This provides a deterministic tie-breaker for the priority queue (heapq), preventing type-comparison errors during routing.

3. Model Extension: Variable Walking Speeds
To accommodate varied walking speeds, I would introduce a speed_factor attribute to the Commuter class. During the shortest path calculation, whenever the algorithm traverses a "transfer edge" (an edge between distinct nodes sharing the same physical station name), it would multiply the baseline transfer time by this commuter-specific factor.


Our current implementation facilitates this perfectly through Loose Coupling. Because commuter behaviors are strictly encapsulated within the Commuter object and kept separate from the TubeNetwork, we can easily individualize journey times without altering the underlying physical graph topology.